In [1]:
# uncomment the following code to redownload the weather data for Waterloo, Ontario for the year 2025. 
# The data will be saved as a CSV file in the "data" folder. 

'''
import requests
import pandas as pd
from pathlib import Path

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 43.4643,
    "longitude": -80.5204,
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "dew_point_2m",
        "apparent_temperature",
        "precipitation",
        "rain",
        "snowfall",
        "weather_code",
        "surface_pressure",
        "cloud_cover",
        "wind_speed_10m",
        "wind_direction_10m",
        "wind_gusts_10m"
    ],
    "timezone": "America/Toronto",
    "temperature_unit": "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm"
}

print("Fetching data from Open-Meteo...")

response = requests.get(url, params=params, timeout=60)

response.raise_for_status()

data = response.json()

df = pd.DataFrame(data["hourly"])

df["latitude"] = data["latitude"]
df["longitude"] = data["longitude"]
df["elevation"] = data["elevation"]
df["timezone"] = data["timezone"]

df["time"] = pd.to_datetime(df["time"])


output_folder = Path("data")
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "waterloo_weather_2025.csv"

df.to_csv(output_file, index=False)


print("\nData downloaded successfully!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Saved to: {output_file.resolve()}")

print("\nFirst five rows:")
print(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

'''

'\nimport requests\nimport pandas as pd\nfrom pathlib import Path\n\nurl = "https://archive-api.open-meteo.com/v1/archive"\n\nparams = {\n    "latitude": 43.4643,\n    "longitude": -80.5204,\n    "start_date": "2025-01-01",\n    "end_date": "2025-12-31",\n    "hourly": [\n        "temperature_2m",\n        "relative_humidity_2m",\n        "dew_point_2m",\n        "apparent_temperature",\n        "precipitation",\n        "rain",\n        "snowfall",\n        "weather_code",\n        "surface_pressure",\n        "cloud_cover",\n        "wind_speed_10m",\n        "wind_direction_10m",\n        "wind_gusts_10m"\n    ],\n    "timezone": "America/Toronto",\n    "temperature_unit": "celsius",\n    "wind_speed_unit": "kmh",\n    "precipitation_unit": "mm"\n}\n\nprint("Fetching data from Open-Meteo...")\n\nresponse = requests.get(url, params=params, timeout=60)\n\nresponse.raise_for_status()\n\ndata = response.json()\n\ndf = pd.DataFrame(data["hourly"])\n\ndf["latitude"] = data["latitude"]\ndf

In [2]:
import sys
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from dotenv import load_dotenv

# Set up project paths so the notebook can access our Python classes.
project_root = Path.cwd().resolve()

if not (project_root / "src").is_dir():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load database settings and import the streaming classes.
load_dotenv(project_root / ".env")

from src.weather_stream import WeatherDatabase, WeatherStreamer

csv_path = project_root / "data" / "waterloo_weather_2025.csv"

print("Project root:", project_root)
print("CSV exists:", csv_path.exists())
print("Setup complete!")

Project root: C:\Users\sybun\OneDrive\Desktop\Python Projects Class\Foundations of ML Frameworks\KNN-Active-Learning-Workshop
CSV exists: True
Setup complete!


In [3]:
# Load the CSV and check its structure and missing values.
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
display(df.head())

print("\nMissing values:")
display(df.isna().sum())

Dataset shape: (8760, 18)


,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,weather_code,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m,latitude,longitude,elevation,timezone
0,2025-01-01 00:00:00,0.7,96,0.2,-3.8,1.0,0.1,0.63,73,962.1,100,16.3,6,32.4,43.479786,-80.54056,324.0,America/Toronto
1,2025-01-01 01:00:00,0.6,96,0.1,-4.1,1.0,0.0,0.70,73,962.2,100,17.6,354,32.4,43.479786,-80.54056,324.0,America/Toronto
2,2025-01-01 02:00:00,0.6,95,-0.1,-4.4,0.8,0.1,0.49,73,962.4,100,19.6,343,36.0,43.479786,-80.54056,324.0,America/Toronto
3,2025-01-01 03:00:00,0.6,94,-0.3,-4.7,0.7,0.1,0.42,73,962.7,100,20.9,339,38.9,43.479786,-80.54056,324.0,America/Toronto
4,2025-01-01 04:00:00,0.6,94,-0.4,-4.9,0.3,0.0,0.21,73,963.3,100,22.2,334,41.4,43.479786,-80.54056,324.0,America/Toronto



Missing values:


time                    0
temperature_2m          0
relative_humidity_2m    0
dew_point_2m            0
apparent_temperature    0
precipitation           0
rain                    0
snowfall                0
weather_code            0
surface_pressure        0
cloud_cover             0
wind_speed_10m          0
wind_direction_10m      0
wind_gusts_10m          0
latitude                0
longitude               0
elevation               0
timezone                0
dtype: int64

In [4]:
# Connect to Neon and display the latest saved weather records.
db = WeatherDatabase()

try:
    db.connect()

    print("Connected to Neon!")
    print("Rows currently stored:", db.count_rows())

    records = db.fetch_latest(limit=10)
    display(pd.DataFrame(records))

finally:
    db.close()

Connected to Neon!
Rows currently stored: 7


,id,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,weather_code,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m,latitude,longitude,elevation,timezone,streamed_at
0,1,2025-01-01 00:00:00,0.7,96.0,0.2,-3.8,1.0,0.1,0.63,73,962.1,100.0,16.3,6.0,32.4,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 01:30:29.514906
1,2,2025-01-01 01:00:00,0.6,96.0,0.1,-4.1,1.0,0.0,0.70,73,962.2,100.0,17.6,354.0,32.4,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 01:30:31.668553
2,3,2025-01-01 02:00:00,0.6,95.0,-0.1,-4.4,0.8,0.1,0.49,73,962.4,100.0,19.6,343.0,36.0,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:44:47.239902
3,4,2025-01-01 03:00:00,0.6,94.0,-0.3,-4.7,0.7,0.1,0.42,73,962.7,100.0,20.9,339.0,38.9,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:44:49.394646
4,5,2025-01-01 04:00:00,0.6,94.0,-0.4,-4.9,0.3,0.0,0.21,73,963.3,100.0,22.2,334.0,41.4,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:44:51.451956
5,6,2025-01-01 05:00:00,0.4,93.0,-0.6,-5.3,0.2,0.0,0.14,71,963.7,100.0,23.9,326.0,44.6,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:44:53.511550
6,7,2025-01-01 06:00:00,0.2,93.0,-0.8,-5.6,0.2,0.0,0.14,71,964.2,100.0,24.3,323.0,45.7,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:44:55.570564


In [5]:
# Resume streaming from the latest saved timestamp.
db = WeatherDatabase()

try:
    db.create_table()
    resume_from = db.last_streamed_time()

    # Send one CSV record to Neon every two seconds.
    streamer = WeatherStreamer(
        csv_path=str(csv_path),
        interval=2
    )

    streamer.subscribe(db.insert_reading)

    print("Starting 5-record stream...")

    sent = streamer.run(
        start_after=resume_from,
        limit=5
    )

    print(f"Streamed {sent} records.")
    print("Total records in Neon:", db.count_rows())

finally:
    db.close()

Starting 5-record stream...
Streamed 5 records.
Total records in Neon: 12


In [9]:
import time
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

# Create four interactive weather charts.
fig = go.FigureWidget(
    make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "Temperature (°C)",
            "Humidity (%)",
            "Wind Speed (km/h)",
            "Precipitation (mm)"
        )
    )
)

weather_columns = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation"
]

for i, column in enumerate(weather_columns):
    fig.add_trace(
        go.Scatter(
            x=[],
            y=[],
            mode="lines+markers",
            name=column
        ),
        row=(i // 2) + 1,
        col=(i % 2) + 1
    )

fig.update_layout(
    title="Waterloo Weather — Live Streaming Dashboard",
    height=700,
    showlegend=False
)

display(fig)

# Load the latest saved records and update the charts.
def refresh_dashboard():
    records = db.fetch_latest(limit=100)
    data = pd.DataFrame(records)

    if data.empty:
        return

    with fig.batch_update():
        for i, column in enumerate(weather_columns):
            fig.data[i].x = data["time"].tolist()
            fig.data[i].y = data[column].tolist()


# Stream new CSV records into Neon and refresh after each insert.
db = WeatherDatabase()

try:
    db.connect()
    resume_from = db.last_streamed_time()

    streamer = WeatherStreamer(
        csv_path=str(csv_path),
        interval=2
    )

    streamer.subscribe(db.insert_reading)
    streamer.subscribe(lambda record: refresh_dashboard())

    refresh_dashboard()

    print("Streaming weather data and updating charts...")

    sent = streamer.run(
        start_after=resume_from,
        limit=10
    )

    print(f"Finished streaming {sent} records.")
    print("Total records in Neon:", db.count_rows())

finally:
    db.close()

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'temperature_2m',
              'type': 'scatter',
              'uid': 'df968ec4-2d17-4c53-9f62-63080b24b438',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'mode': 'lines+markers',
              'name': 'relative_humidity_2m',
              'type': 'scatter',
              'uid': '167573b7-66d9-40b4-b968-07fbae94acdb',
              'x': [],
              'xaxis': 'x2',
              'y': [],
              'yaxis': 'y2'},
             {'mode': 'lines+markers',
              'name': 'wind_speed_10m',
              'type': 'scatter',
              'uid': 'd5c59b23-d4bd-49aa-82d2-b81644532a35',
              'x': [],
              'xaxis': 'x3',
              'y': [],
              'yaxis': 'y3'},
             {'mode': 'lines+markers',
              'name': 'precipitation',
              'type': 'scatter',
              'uid': '01d103f2-cbb3-

Streaming weather data and updating charts...
Finished streaming 10 records.
Total records in Neon: 32
